In [1]:
import math
import os
import random
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from scipy.optimize import minimize
from scipy.stats import spearmanr
from tqdm.auto import tqdm

# Walk up from the notebook folder to locate the repository's `code/` root
CODE_DIR = next(
    (p for p in (Path.cwd(), *Path.cwd().parents) if all((p / d).is_dir() for d in ("kev", "data", "training"))),
    None,
)
if CODE_DIR is None:
    raise RuntimeError(f"No code/ root containing kev/, data/, and training/ at or above {Path.cwd()}")

# Run from `code/` and expose both `code/` and `code/kev/` on sys.path
os.chdir(CODE_DIR)
for _p in (str(CODE_DIR), str(CODE_DIR / "kev")):
    if _p not in sys.path:
        sys.path.insert(0, _p)

from data.base import (
    build_long_records,
    fused_eval_context,
    group_records_by_state,
    make_grouped_view,
    make_view,
    predict_probs_grouped,
)
from data.wu2025 import WU2025_PERSONA_COLS, load_wu2025_df
from data.wu2025.stimulus import ATTR_LOCAL, ATTR_MONEY, ATTR_PERCENT, CONJOINT_OPTIONS
from models.kev_pointer import KevPointerModel
from training import free_gpu

print(f"CODE_DIR: {CODE_DIR}")

CODE_DIR: /home/dw/github/decision-models-finetuning/code


In [2]:
# Load all included Wu 2025 survey rows and filter to paired electricity-plan conjoint tasks (`plan_choice`)
t_load0 = time.perf_counter()
wu2025_df = load_wu2025_df()
plan_choice_df = wu2025_df[wu2025_df["v"] == "plan_choice"].reset_index(drop=True)
load_wu_sec = time.perf_counter() - t_load0
if len(plan_choice_df) == 0:
    raise ValueError("No plan_choice rows found in wu2025_df")

# Summarize dataset size and load time: each answered task contributes 2 option rows to OLS
display(
    pd.Series(
        {
            "wu2025_total_rows": len(wu2025_df),
            "wu2025_questions": wu2025_df["v"].nunique(),
            "plan_choice_tasks": len(plan_choice_df),
            "plan_choice_option_rows": len(plan_choice_df) * 2,
            "respondents_with_plan_choice": plan_choice_df["CASEID"].nunique(),
            "distinct_displayed_pairs": plan_choice_df["qid"].nunique(),
            "load_sec": round(load_wu_sec, 4),
        },
        name="wu2025_overview",
    ).to_frame()
)

,wu2025_overview
wu2025_total_rows,90979.000
wu2025_questions,46.000
plan_choice_tasks,18591.000
plan_choice_option_rows,37182.000
respondents_with_plan_choice,1861.000
distinct_displayed_pairs,8212.000
load_sec,0.197


In [3]:
# Ordered factor levels per attribute; index 0 in each list is the OLS reference level ($105, 10%, No)
WU2025_LEVELS = {
    ATTR_MONEY: ["$105", "$110", "$115", "$120", "$125"],
    ATTR_PERCENT: ["10%", "30%", "50%", "100%"],
    ATTR_LOCAL: ["No", "Yes"],
}

# Ordered names for the 8 non-intercept AMCE coefficients matching columns 1..8 of X
WU2025_BETA_NAMES = [
    *(f"{ATTR_MONEY}: {lvl} vs $105" for lvl in WU2025_LEVELS[ATTR_MONEY][1:]),
    *(f"{ATTR_PERCENT}: {lvl} vs 10%" for lvl in WU2025_LEVELS[ATTR_PERCENT][1:]),
    *(f"{ATTR_LOCAL}: {lvl} vs No" for lvl in WU2025_LEVELS[ATTR_LOCAL][1:]),
]

# SVD relative singular-value cutoff used to verify full column rank before inverting X
OLS_SVD_RCOND = 1e-8


def parse_wu2025_plan_description(desc: str) -> dict[str, str]:
    # Attribute display order is randomized per respondent, so split into 3 `Attribute: Level` clauses
    parts = desc.split("; ")
    if len(parts) != 3:
        raise ValueError(f"Expected 3 attribute clauses, got {desc!r}")
    kv: dict[str, str] = {}
    for part in parts:
        attr, sep, level = part.partition(": ")
        if not sep:
            raise ValueError(f"Malformed attribute clause {part!r} in {desc!r}")
        # Fail fast if any attribute name or level deviates from the known experimental design
        if attr not in WU2025_LEVELS:
            raise KeyError(f"Unexpected attribute {attr!r} in {desc!r}")
        if level not in WU2025_LEVELS[attr]:
            raise ValueError(f"Unexpected level {level!r} for {attr!r}")
        kv[attr] = level
    if set(kv.keys()) != set(WU2025_LEVELS.keys()):
        raise ValueError(f"Missing attributes in {desc!r}: got {sorted(kv.keys())}")
    return kv


def encode_wu2025_x_row(desc: str) -> list[float]:
    # Convert one option's description into [1.0 (intercept), 4 bill dummies, 3 renewables-% dummies, 1 local dummy]
    parsed = parse_wu2025_plan_description(desc)
    return [
        1.0,  # col 0: constant intercept (absorbs baseline choice level, dropped after projection)
        *[1.0 if parsed[ATTR_MONEY] == lvl else 0.0 for lvl in WU2025_LEVELS[ATTR_MONEY][1:]],      # cols 1..4: $110, $115, $120, $125
        *[1.0 if parsed[ATTR_PERCENT] == lvl else 0.0 for lvl in WU2025_LEVELS[ATTR_PERCENT][1:]],  # cols 5..7: 30%, 50%, 100%
        *[1.0 if parsed[ATTR_LOCAL] == lvl else 0.0 for lvl in WU2025_LEVELS[ATTR_LOCAL][1:]],      # col 8: Yes
    ]


def build_ols_projector(X: torch.Tensor) -> torch.Tensor:
    # Build fixed linear OLS operator A of shape (K, N) mapping N option outcomes -> K non-intercept betas
    if X.ndim != 2 or X.shape[0] < X.shape[1] or X.shape[1] < 2:
        raise ValueError(f"Expected 2D full-column design matrix X with N >= P >= 2, got shape {tuple(X.shape)}")
    X64 = X.to(dtype=torch.float64)
    # Fail fast via SVD if X is rank-deficient
    svals = torch.linalg.svdvals(X64)
    if float(svals[-1]) <= OLS_SVD_RCOND * float(svals[0]):
        raise ValueError(f"Rank-deficient design matrix X: min_sval={float(svals[-1]):.3e}, max_sval={float(svals[0]):.3e}")
    # pinv(X64) computes (X^T X)^{-1} X^T via SVD without squaring the condition number; [1:] drops the intercept
    return torch.linalg.pinv(X64)[1:]


def fit_ols_betas(A: torch.Tensor, outcome: torch.Tensor) -> torch.Tensor:
    # Linear OLS projection: beta = A @ outcome (differentiable whenever `outcome` comes from model predictions)
    if outcome.ndim != 1 or outcome.shape[0] != A.shape[1]:
        raise ValueError(f"Expected 1D outcome of length {A.shape[1]}, got shape {tuple(outcome.shape)}")
    return A @ outcome.to(dtype=A.dtype)


def build_ols_inputs(
    tasks_df: pd.DataFrame,
    device: str,
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    # Validate every 2-option task row and build stacked (2*T, 9) X, (2*T,) targets y, and (8, 2*T) projector A
    if len(tasks_df) == 0:
        raise ValueError("Cannot build OLS inputs from an empty DataFrame")
    x_rows: list[list[float]] = []
    y_vals: list[float] = []
    for task_idx, row in enumerate(tasks_df.itertuples(index=False)):
        if list(row.option_texts) != CONJOINT_OPTIONS:
            raise ValueError(f"Task {task_idx}: expected options {CONJOINT_OPTIONS}, got {row.option_texts}")
        if row.option_descriptions is None or len(row.option_descriptions) != 2:
            raise ValueError(f"Task {task_idx}: expected 2 option_descriptions, got {row.option_descriptions}")
        one_hot = [float(v) for v in row.one_hot]
        if len(one_hot) != 2 or sorted(one_hot) != [0.0, 1.0]:
            raise ValueError(f"Task {task_idx}: expected binary one_hot of length 2, got {one_hot}")
        for desc, y_val in zip(row.option_descriptions, one_hot, strict=True):
            x_rows.append(encode_wu2025_x_row(desc))
            y_vals.append(y_val)
    X_mat = torch.tensor(x_rows, dtype=torch.float64)
    y_flat = torch.tensor(y_vals, dtype=torch.float64, device=device)
    A_mat = build_ols_projector(X_mat).to(device)
    return X_mat, y_flat, A_mat


print(f"Configured {len(WU2025_BETA_NAMES)} non-intercept OLS beta terms (+ 1 intercept = {len(WU2025_BETA_NAMES) + 1} design columns)")

Configured 8 non-intercept OLS beta terms (+ 1 intercept = 9 design columns)


### Expected OLS Beta Target Noise MSE vs Respondent `sample_size` per Step

Each optimization step draws a sample of `sample_size` human respondents totaling $T = \sum_{r=1}^{\text{sample\_size}} T_r$ binary forced-choice tasks ($2T$ option rows). On task $t$, the respondent selects either `Choice 1` or `Choice 2`, so $y_{2t-1} = y_{t,1} \in \{0, 1\}$ and $y_{2t} = 1 - y_{t,1}$, producing a paired Bernoulli residual $\varepsilon_t = y_{t,1} - \pi_{t,1}$ with variance $v_t = \pi_{t,1}(1 - \pi_{t,1})$ that enters the two option rows as $\varepsilon_t [+1, -1]^\top$.

For the respondent sample's stacked design matrix $X_{\text{sample}} \in \mathbb{R}^{2T \times 9}$ and non-intercept OLS projector $A_{\text{sample}} = (X_{\text{sample}}^\dagger)_{1:,:} \in \mathbb{R}^{8 \times 2T}$, define the $8 \times T$ task-sensitivity matrix $A_{\text{diff}} = A_{\text{sample}}[:, 0::2] - A_{\text{sample}}[:, 1::2]$. The sampling error of the human OLS target $\hat{\beta}_{\text{human}}^{\text{sample}} = A_{\text{sample}} y_{\text{sample}}$ around its noiseless expectation $\beta^*_{\text{sample}} = A_{\text{sample}} \pi_{\text{sample}}$ is $A_{\text{diff}} \varepsilon$, giving the exact expected 8-coefficient target noise MSE:

$$\text{Noise MSE}(\text{sample\_size}) = \frac{1}{8}\operatorname{Tr}\!\left(A_{\text{diff}} \operatorname{diag}(v) A_{\text{diff}}^\top\right) = \frac{1}{8}\sum_{k=1}^{8}\sum_{t=1}^{T} A_{\text{diff}}[k, t]^2 \, \pi_{t,1}(1 - \pi_{t,1})$$

In [5]:
# Compute full-population OLS projector `A_pop` (8 x 37,182) and ground-truth human AMCE betas `human_pop_betas`
t_noise0 = time.perf_counter()
X_pop, y_pop, A_pop = build_ols_inputs(plan_choice_df, device="cpu")
human_pop_betas = fit_ols_betas(A_pop, y_pop)
if bool((human_pop_betas == 0).any()):
    raise ValueError("Zero population human OLS beta encountered; MAPE denominator would be undefined")
SIGNAL_BETA_MS = float(human_pop_betas.pow(2).mean())  # mean square (signal power) (1/8)*||beta_human_pop||_2^2

# Fit a 9-parameter paired conditional logit (Choice 1 bias + 8 attribute utilities) to estimate task Bernoulli variances v_t
X_pop_np = X_pop.numpy()
X_task_diff = np.column_stack([np.ones(len(plan_choice_df)), X_pop_np[0::2, 1:] - X_pop_np[1::2, 1:]])
y_choice1_np = y_pop.numpy()[0::2]


def _paired_logit_nll_and_grad(w: np.ndarray) -> tuple[float, np.ndarray]:
    # Negative log-likelihood and analytical gradient for P(Choice 1 | task t) = sigmoid(X_task_diff @ w)
    z = X_task_diff @ w
    p1 = 1.0 / (1.0 + np.exp(-z))
    nll = float(-np.mean(y_choice1_np * np.log(np.clip(p1, 1e-15, 1.0)) + (1.0 - y_choice1_np) * np.log(np.clip(1.0 - p1, 1e-15, 1.0))))
    grad = (X_task_diff.T @ (p1 - y_choice1_np)) / len(y_choice1_np)
    return nll, grad


logit_fit = minimize(_paired_logit_nll_and_grad, np.zeros(X_task_diff.shape[1]), jac=True, method="BFGS")
if not logit_fit.success:
    raise RuntimeError(f"Paired conditional logit MLE failed to converge: {logit_fit.message}")
pi_choice1 = 1.0 / (1.0 + np.exp(-(X_task_diff @ logit_fit.x)))
task_bernoulli_var = pi_choice1 * (1.0 - pi_choice1)  # v_t = pi_{t,1} * (1 - pi_{t,1}) for each of the 18,591 tasks

# Slice per-respondent design matrices `X_r` and per-task Bernoulli variances `v_r` in deterministic epoch shuffle order
respondent_noise_items: list[tuple[torch.Tensor, torch.Tensor]] = []
row_cursor = 0
task_cursor = 0
for _, r_group in plan_choice_df.groupby("CASEID", sort=False):
    n_tasks = len(r_group)
    X_r = X_pop[row_cursor : row_cursor + 2 * n_tasks]
    v_r = torch.from_numpy(task_bernoulli_var[task_cursor : task_cursor + n_tasks]).to(dtype=torch.float64)
    respondent_noise_items.append((X_r, v_r))
    row_cursor += 2 * n_tasks
    task_cursor += n_tasks

if row_cursor != X_pop.shape[0] or task_cursor != len(plan_choice_df):
    raise AssertionError("Row/task cursor mismatch while slicing respondent design blocks")


def analyze_sample_size_noise(sample_size: int, seed: int) -> dict[str, float | int]:
    # Shuffle respondents with the training epoch seed and group into consecutive human respondent samples of `sample_size`
    if not isinstance(sample_size, int) or isinstance(sample_size, bool) or sample_size < 1:
        raise ValueError(f"sample_size must be a positive integer, got {sample_size!r}")
    shuffled = list(respondent_noise_items)
    random.Random(seed).shuffle(shuffled)

    total_steps = math.ceil(len(shuffled) / sample_size)
    rank_def_steps = 0
    option_row_counts: list[int] = []
    step_noise_mses: list[float] = []
    step_op_norms: list[float] = []

    for start in range(0, len(shuffled), sample_size):
        sample_slice = shuffled[start : start + sample_size]
        X_sample = torch.cat([it[0] for it in sample_slice], dim=0)
        v_sample = torch.cat([it[1] for it in sample_slice], dim=0)
        option_row_counts.append(int(X_sample.shape[0]))

        # Check full column rank 9/9 on the pooled respondent sample before computing A_sample = pinv(X_sample)[1:]
        if X_sample.shape[0] >= X_sample.shape[1]:
            svals = torch.linalg.svdvals(X_sample)
            is_full_rank = float(svals[-1]) > OLS_SVD_RCOND * float(svals[0])
        else:
            is_full_rank = False

        if not is_full_rank:
            rank_def_steps += 1
            continue

        A_sample = torch.linalg.pinv(X_sample)[1:]
        A_diff = A_sample[:, 0::2] - A_sample[:, 1::2]  # (8, T_sample) paired-choice sensitivity operator
        noise_mse = float(((A_diff.pow(2)) * v_sample.unsqueeze(0)).sum(dim=1).mean())
        op_norm = float(torch.linalg.svdvals(A_diff)[0])
        step_noise_mses.append(noise_mse)
        step_op_norms.append(op_norm)

    mean_noise_mse = float(np.mean(step_noise_mses))
    return {
        "sample_size": sample_size,
        "steps_per_epoch": total_steps,
        "mean_option_rows": float(np.mean(option_row_counts)),
        "rank_deficient_steps": rank_def_steps,
        "signal_beta_ms": SIGNAL_BETA_MS,
        "target_noise_mse": mean_noise_mse,
        "target_noise_rmse": math.sqrt(mean_noise_mse),
        "snr_ratio": SIGNAL_BETA_MS / mean_noise_mse,
        "mean_A_diff_op_norm": float(np.mean(step_op_norms)),
    }


SAMPLE_SIZES_TO_ANALYZE = (1, 2, 4, 8, 16, 32, 64, len(respondent_noise_items))
sample_size_noise_df = pd.DataFrame(
    [analyze_sample_size_noise(sample_size=s, seed=42) for s in SAMPLE_SIZES_TO_ANALYZE]
).set_index("sample_size")
noise_analysis_sec = time.perf_counter() - t_noise0

print(
    f"Ground-truth population OLS signal mean square ||beta_pop||^2 / 8 = {SIGNAL_BETA_MS:.5f} "
    f"(RMS = {math.sqrt(SIGNAL_BETA_MS):.4f}) | analysis_sec = {noise_analysis_sec:.3f}s"
)
display(pd.Series(human_pop_betas.numpy(), index=WU2025_BETA_NAMES, name="human_pop_beta").round(4).to_frame())
display(sample_size_noise_df.round(5))

Ground-truth population OLS signal mean square ||beta_pop||^2 / 8 = 0.04746 (RMS = 0.2179) | analysis_sec = 0.831s


,human_pop_beta
Monthly bill: $110 vs $105,-0.0969
Monthly bill: $115 vs $105,-0.1779
Monthly bill: $120 vs $105,-0.2894
Monthly bill: $125 vs $105,-0.3832
Percentage of renewables: 30% vs 10%,0.0973
Percentage of renewables: 50% vs 10%,0.1706
Percentage of renewables: 100% vs 10%,0.2421
Locally produced renewables: Yes vs No,0.1042


,steps_per_epoch,mean_option_rows,rank_deficient_steps,signal_beta_ms,target_noise_mse,target_noise_rmse,snr_ratio,mean_A_diff_op_norm
sample_size,,,,,,,,
1,1861,19.97958,136,0.04746,0.14934,0.38645,0.31782,1.77390
2,931,39.93770,0,0.04746,0.05083,0.22544,0.93387,0.97272
4,466,79.78970,0,0.04746,0.02190,0.14798,2.16737,0.61594
8,233,159.57940,0,0.04746,0.01021,0.10104,4.64877,0.41626
16,117,317.79487,0,0.04746,0.00504,0.07100,9.41586,0.28907
32,59,630.20339,0,0.04746,0.00269,0.05185,17.65613,0.20605
64,30,1239.40000,0,0.04746,0.00176,0.04191,27.02872,0.15361
1861,1,37182.00000,0,0.04746,0.00004,0.00642,1152.16402,0.02606


In [6]:
def compute_wu2025_loss(
    task_logits: torch.Tensor,
    y_flat: torch.Tensor,
    A: torch.Tensor,
    beta_lambda: float,
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    # Validate paired 2-option task logits against flattened sample option targets and sample OLS projector A
    if task_logits.ndim != 2 or task_logits.shape[1] != 2:
        raise ValueError(f"Expected (num_tasks, 2) task_logits, got shape {tuple(task_logits.shape)}")
    num_rows = task_logits.numel()
    if y_flat.shape != (num_rows,) or A.shape[1] != num_rows:
        raise ValueError(f"Shape mismatch: num_rows={num_rows}, y_flat={tuple(y_flat.shape)}, A={tuple(A.shape)}")
    if y_flat.requires_grad or A.requires_grad:
        raise ValueError("Human choice targets `y_flat` and OLS projector `A` must be fixed (requires_grad=False)")

    # 1. Per-response KL divergence KL(y || p) (= CE on one-hot choices) averaged across all tasks in the respondent sample
    log_probs = torch.log_softmax(task_logits, dim=-1)
    y_2col = y_flat.view(-1, 2)
    kl_loss = torch.nn.functional.kl_div(log_probs, y_2col, reduction="batchmean")

    # 2. When beta_lambda == 0, compute sample OLS beta MSE under no_grad for logging so backward() stays pure KL/CE
    if beta_lambda == 0.0:
        with torch.no_grad():
            pred_probs_detached = torch.softmax(task_logits.detach(), dim=-1).reshape(-1)
            beta_loss = (fit_ols_betas(A, pred_probs_detached) - fit_ols_betas(A, y_flat)).pow(2).mean()
        return kl_loss, kl_loss, beta_loss

    # 3. Differentiable sample-level OLS beta MSE loss: mean((A_sample @ pred_probs - A_sample @ y_sample)^2)
    pred_probs = torch.softmax(task_logits, dim=-1).reshape(-1)
    pred_betas = fit_ols_betas(A, pred_probs)
    human_betas = fit_ols_betas(A, y_flat)
    beta_loss = (pred_betas - human_betas).pow(2).mean()

    # 4. Combined single-pass objective over the respondent sample: choice KL loss + beta_lambda * sample OLS beta MSE
    total_loss = kl_loss + beta_lambda * beta_loss
    return total_loss, kl_loss, beta_loss


print("Defined compute_wu2025_loss(task_logits, y_flat, A, beta_lambda)")

Defined compute_wu2025_loss(task_logits, y_flat, A, beta_lambda)


In [7]:
# Require CUDA and release any previously loaded `kev_model` before allocating a fresh 800M backbone
if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU is required to run jaredpalmer/kev-0.8b in bf16 train mode")
if "kev_model" in globals():
    globals().pop("kev_model")
device = "cuda"
free_gpu()
torch.manual_seed(42)

# Build `jaredpalmer/kev-0.8b` with the v5 training configuration (bf16 frozen backbone + rank-64 LoRA + pointer head)
t_model0 = time.perf_counter()
kev_model = KevPointerModel.build(
    "jaredpalmer/kev-0.8b",
    device=device,
    lora_rank=64,
    weights_dtype="bf16",
)
kev_model.net.head.temperature = 1.0  # training uses raw uncalibrated temperature = 1.0
kev_model.train()                     # enable training mode (LoRA dropout active, trainable params require_grad)
torch.cuda.synchronize()
model_load_sec = time.perf_counter() - t_model0

# Summarize loaded model state, parameter counts, and GPU load time
trainable_params_count = sum(p.numel() for p in kev_model.trainable_parameters())
total_params_count = sum(p.numel() for p in kev_model.parameters())
display(
    pd.Series(
        {
            "base_model": "jaredpalmer/kev-0.8b",
            "mode": kev_model.mode,
            "training": kev_model.training,
            "weights_dtype": kev_model.meta.weights_dtype,
            "lora_rank": kev_model.meta.lora,
            "trainable_params": f"{trainable_params_count:,}",
            "total_params": f"{total_params_count:,}",
            "model_load_sec": round(model_load_sec, 3),
        },
        name="kev_800m_train_mode",
    ).to_frame()
)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

,kev_800m_train_mode
base_model,jaredpalmer/kev-0.8b
mode,lora_rank 64
training,True
weights_dtype,bf16
lora_rank,64
trainable_params,"43,815,424"
total_params,"796,208,448"
model_load_sec,6.756


In [8]:
# Build state-grouped evaluation bundle across all 18,591 Wu 2025 `plan_choice` tasks (1,831 unique persona states)
t_pop0 = time.perf_counter()
wu2025_pop_records = build_long_records(
    plan_choice_df,
    persona_cols=WU2025_PERSONA_COLS,
    include_country=False,
    year="wu2025",
)
wu2025_pop_bundle = group_records_by_state(
    wu2025_pop_records,
    view_fn=make_view,
    name="wu2025_plan_choice",
    valid=True,
)
pop_bundle_prep_sec = time.perf_counter() - t_pop0

# Fail fast if evaluation bundle record count does not match the precomputed population OLS projector `A_pop`
if len(wu2025_pop_bundle["records"]) * 2 != A_pop.shape[1]:
    raise AssertionError(
        f"Eval bundle tasks ({len(wu2025_pop_bundle['records'])}) * 2 != A_pop columns ({A_pop.shape[1]})"
    )

print(
    f"Population eval bundle: {len(wu2025_pop_bundle['records']):,} tasks | "
    f"{len(wu2025_pop_bundle['packed_views']):,} packed persona states | "
    f"A_pop shape: {tuple(A_pop.shape)} | prep_sec: {pop_bundle_prep_sec:.3f}s"
)

Population eval bundle: 18,591 tasks | 1,831 packed persona states | A_pop shape: (8, 37182) | prep_sec: 0.423s


In [9]:
@torch.no_grad()
def validate_wu2025_population_ols(
    model: KevPointerModel,
    pop_bundle: dict,
    A_pop: torch.Tensor,
    y_pop: torch.Tensor,
    human_pop_betas: torch.Tensor,
) -> tuple[dict[str, float], pd.DataFrame]:
    # Time full evaluation wall-clock (including LoRA fusion / CudaGraphs setup + GPU inference + OLS projection)
    torch.cuda.synchronize()
    t_eval_wall0 = time.perf_counter()

    # Switch to eval mode and fuse current LoRA deltas + CudaGraphs for shared-state inference across all respondents
    was_training = model.training
    model.eval()
    free_gpu()
    try:
        with torch.autocast("cuda", dtype=torch.bfloat16), fused_eval_context(model, eval_dtype=torch.bfloat16):
            preds, stats = predict_probs_grouped(model, pop_bundle, show_progress=False, return_stats=True)
    finally:
        # Always restore training mode and release temporary graph buffers so training can resume immediately
        if was_training:
            model.train()
        free_gpu()

    # Fail fast if fused_eval_context failed to restore trainable parameter state or training mode
    if model.training != was_training:
        raise AssertionError(f"Expected model.training={was_training} after validation, got {model.training}")
    if any((not p.requires_grad) or p.numel() == 0 for p in model.trainable_parameters()):
        raise AssertionError("Trainable parameters were not cleanly restored after fused_eval_context")

    # Time CPU tensor stacking, population KL computation, and `A_pop @ pred_probs` OLS projection
    t_ols0 = time.perf_counter()
    num_tasks = y_pop.numel() // 2
    if len(preds) != num_tasks:
        raise ValueError(f"Expected {num_tasks} task predictions, got {len(preds)}")
    pred_probs_2col = torch.from_numpy(np.stack(preds, axis=0)).to(dtype=torch.float64)
    if pred_probs_2col.shape != (num_tasks, 2):
        raise ValueError(f"Expected ({num_tasks}, 2) predicted probabilities, got {tuple(pred_probs_2col.shape)}")

    pop_kl_loss = float(
        torch.nn.functional.kl_div(
            pred_probs_2col.clamp_min(1e-12).log(),
            y_pop.view(-1, 2),
            reduction="batchmean",
        )
    )
    model_pop_betas = fit_ols_betas(A_pop, pred_probs_2col.reshape(-1))
    beta_diff = model_pop_betas - human_pop_betas
    eval_ols_ms = (time.perf_counter() - t_ols0) * 1000.0
    eval_wall_sec = time.perf_counter() - t_eval_wall0

    # Report statistical OLS / KL metrics side-by-side with detailed evaluation timing metrics
    metrics = {
        "pop_kl_loss": pop_kl_loss,
        "pop_beta_mse": float(beta_diff.pow(2).mean()),
        "pop_beta_mae": float(beta_diff.abs().mean()),
        "pop_beta_mape_pct": float((beta_diff.abs() / human_pop_betas.abs()).mean() * 100.0),
        "spearman_rho": float(spearmanr(model_pop_betas.numpy(), human_pop_betas.numpy()).statistic),
        "eval_prep_sec": float(stats["prep_sec"]),
        "eval_gpu_sec": float(stats["gpu_sec"]),
        "eval_ols_ms": eval_ols_ms,
        "eval_wall_sec": eval_wall_sec,
        "eval_ms_per_task": (eval_wall_sec * 1000.0) / num_tasks,
    }
    beta_df = pd.DataFrame(
        {
            "human_pop_beta": human_pop_betas.numpy(),
            "model_pop_beta": model_pop_betas.numpy(),
            "diff": beta_diff.numpy(),
            "abs_diff": beta_diff.abs().numpy(),
        },
        index=pd.Index(WU2025_BETA_NAMES, name="term"),
    )
    return metrics, beta_df


# Run Step 0 baseline validation on untrained `jaredpalmer/kev-0.8b` before any training updates
step0_val_metrics, step0_beta_df = validate_wu2025_population_ols(
    kev_model,
    wu2025_pop_bundle,
    A_pop,
    y_pop,
    human_pop_betas,
)
display(pd.Series(step0_val_metrics, name="step_0_population_val").round(4).to_frame())
display(step0_beta_df.round(4))

,step_0_population_val
pop_kl_loss,1.0699
pop_beta_mse,0.0512
pop_beta_mae,0.2093
pop_beta_mape_pct,114.1314
spearman_rho,-0.2143
eval_prep_sec,0.2057
eval_gpu_sec,33.3829
eval_ols_ms,5.3574
eval_wall_sec,35.4998
eval_ms_per_task,1.9095


,human_pop_beta,model_pop_beta,diff,abs_diff
term,,,,
Monthly bill: $110 vs $105,-0.0969,-0.0068,0.0901,0.0901
Monthly bill: $115 vs $105,-0.1779,-0.0122,0.1658,0.1658
Monthly bill: $120 vs $105,-0.2894,-0.0171,0.2723,0.2723
Monthly bill: $125 vs $105,-0.3832,-0.0248,0.3584,0.3584
Percentage of renewables: 30% vs 10%,0.0973,-0.0075,-0.1047,0.1047
Percentage of renewables: 50% vs 10%,0.1706,-0.0149,-0.1856,0.1856
Percentage of renewables: 100% vs 10%,0.2421,-0.0398,-0.2820,0.2820
Locally produced renewables: Yes vs No,0.1042,-0.1109,-0.2151,0.2151


In [10]:
# Snapshot all 376 untrained LoRA + pointer-head tensors on CPU so every `(sample_size, beta_lambda)` run starts identically
INIT_TRAINABLE_STATE = {
    name: param.detach().cpu().clone()
    for name, param in kev_model.named_parameters()
    if param.requires_grad
}


def reset_kev_trainable_weights(model: KevPointerModel, init_state: dict[str, torch.Tensor]) -> None:
    # Fail fast if the model's trainable parameter set changed before copying Step-0 weights back onto GPU
    current_names = {name for name, param in model.named_parameters() if param.requires_grad}
    if current_names != set(init_state.keys()):
        raise ValueError("Trainable parameter names do not match INIT_TRAINABLE_STATE snapshot")
    with torch.no_grad():
        for name, param in model.named_parameters():
            if param.requires_grad:
                param.copy_(init_state[name].to(device=param.device, dtype=param.dtype))
                param.grad = None
    model.train()
    free_gpu()


# Pre-tokenize all 1,861 respondents once, storing `enc`, `X_resp` (CPU float64), and `y_resp` (GPU float64)
# so any respondent `sample_size` can stack `X_sample` and `y_sample` on the fly without re-tokenizing
t_resp_prep0 = time.perf_counter()
wu2025_respondent_items: list[dict] = []
single_resp_full_rank_count = 0

for r_caseid, r_group in tqdm(
    plan_choice_df.groupby("CASEID", sort=False),
    total=plan_choice_df["CASEID"].nunique(),
    desc="Preparing respondents",
):
    r_df = r_group.reset_index(drop=True)
    x_rows: list[list[float]] = []
    y_vals: list[float] = []
    for t_idx, row in enumerate(r_df.itertuples(index=False)):
        # Validate every respondent task's options, descriptions, and binary one-hot choice vector
        if list(row.option_texts) != CONJOINT_OPTIONS:
            raise ValueError(f"{r_caseid} task {t_idx}: expected {CONJOINT_OPTIONS}, got {row.option_texts}")
        if row.option_descriptions is None or len(row.option_descriptions) != 2:
            raise ValueError(f"{r_caseid} task {t_idx}: expected 2 option_descriptions")
        one_hot = [float(v) for v in row.one_hot]
        if len(one_hot) != 2 or sorted(one_hot) != [0.0, 1.0]:
            raise ValueError(f"{r_caseid} task {t_idx}: invalid binary one_hot {one_hot}")
        for desc, y_val in zip(row.option_descriptions, one_hot, strict=True):
            x_rows.append(encode_wu2025_x_row(desc))
            y_vals.append(y_val)

    # Tokenize the respondent's shared persona state + choice tasks once and verify target alignment
    r_records = build_long_records(r_df, persona_cols=WU2025_PERSONA_COLS, include_country=False, year="wu2025")
    r_enc, r_targets = kev_model.prepare(make_grouped_view(r_records))
    r_X = torch.tensor(x_rows, dtype=torch.float64)
    r_y = torch.tensor(y_vals, dtype=torch.float64, device=device)
    prepared_2col = torch.stack([q["target"].to(dtype=torch.float64, device=device) for q in r_targets], dim=0)
    if not torch.equal(prepared_2col, r_y.view(-1, 2)):
        raise AssertionError(f"{r_caseid}: Kev prepared target order does not match y_resp")

    if r_X.shape[0] >= r_X.shape[1]:
        r_svals = torch.linalg.svdvals(r_X)
        if float(r_svals[-1]) > OLS_SVD_RCOND * float(r_svals[0]):
            single_resp_full_rank_count += 1

    wu2025_respondent_items.append(
        {
            "caseid": str(r_caseid),
            "num_tasks": len(r_df),
            "enc": r_enc,
            "X_resp": r_X,
            "y_resp": r_y,
        }
    )

resp_prep_sec = time.perf_counter() - t_resp_prep0
if (len(wu2025_respondent_items), single_resp_full_rank_count) != (1861, 1725):
    raise AssertionError(
        f"Unexpected respondent counts: {(len(wu2025_respondent_items), single_resp_full_rank_count)}"
    )

# Registry keyed by `(sample_size, beta_lambda)` storing full step/timing logs and population OLS trajectories
experiment_runs: dict[tuple[int, float], dict] = {}

display(
    pd.Series(
        {
            "total_respondents": len(wu2025_respondent_items),
            "single_respondent_full_rank": single_resp_full_rank_count,
            "single_respondent_rank_deficient": len(wu2025_respondent_items) - single_resp_full_rank_count,
            "snapshot_trainable_tensors": len(INIT_TRAINABLE_STATE),
            "respondent_prep_sec": round(resp_prep_sec, 3),
            "ms_per_respondent_prep": round((resp_prep_sec * 1000.0) / len(wu2025_respondent_items), 3),
        },
        name="epoch_prep_summary",
    ).to_frame()
)

Preparing respondents:   0%|          | 0/1861 [00:00<?, ?it/s]

,epoch_prep_summary
total_respondents,1861.000
single_respondent_full_rank,1725.000
single_respondent_rank_deficient,136.000
snapshot_trainable_tensors,376.000
respondent_prep_sec,7.360
ms_per_respondent_prep,3.955


In [11]:
# Shared epoch hyperparameters; each optimization step draws a sample of `sample_size` human respondents
EPOCH_SEED = 42
BASE_LR_SAMPLE_1 = 2e-5               # validated 1-respondent/step learning rate from v1
EPOCH_WEIGHT_DECAY = 0.01
NUM_EPOCH_VAL_CHECKS = 4              # validate population OLS ~4 times per epoch (~every quarter of respondents)
FWD_CHUNK_RESPONDENTS = 8             # max respondents batched into one `shared_prefix=True` GPU kernel call
MAX_UNCHECKPOINTED_SAMPLE_SIZE = 8    # enable per-layer activation checkpointing when `sample_size > 8` to bound VRAM


def scale_lr_for_sample_size(sample_size: int) -> float:
    # Square-root AdamW LR scaling keeps 1-epoch parameter travel invariant as `steps_per_epoch` shrinks by 1/sample_size
    if not isinstance(sample_size, int) or isinstance(sample_size, bool) or sample_size < 1:
        raise ValueError(f"sample_size must be a positive integer, got {sample_size!r}")
    return BASE_LR_SAMPLE_1 * math.sqrt(sample_size)


def run_wu2025_epoch(
    sample_size: int,
    beta_lambda: float,
    lr: float,
) -> dict:
    # Fail fast on invalid sample_size, beta_lambda, or learning rate
    if not isinstance(sample_size, int) or isinstance(sample_size, bool) or sample_size < 1:
        raise ValueError(f"sample_size must be a positive integer, got {sample_size!r}")
    if beta_lambda < 0.0:
        raise ValueError(f"beta_lambda must be non-negative, got {beta_lambda}")
    if lr <= 0.0:
        raise ValueError(f"lr must be positive, got {lr}")

    t_wall0 = time.perf_counter()

    # Reset weights to exact Step-0 initialization and re-seed RNGs so respondent sample order matches across runs
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    reset_kev_trainable_weights(kev_model, INIT_TRAINABLE_STATE)
    if sample_size > MAX_UNCHECKPOINTED_SAMPLE_SIZE:
        kev_model.enable_checkpointing()
    else:
        kev_model.net.lm.gradient_checkpointing_disable()

    rng = random.Random(EPOCH_SEED)
    torch.manual_seed(EPOCH_SEED)
    torch.cuda.manual_seed_all(EPOCH_SEED)

    epoch_items = list(wu2025_respondent_items)
    rng.shuffle(epoch_items)
    total_respondents = len(epoch_items)
    total_steps = math.ceil(total_respondents / sample_size)
    val_every = max(1, total_steps // NUM_EPOCH_VAL_CHECKS)
    postfix_every = max(1, total_steps // 60)
    rolling_steps = max(1, round(300 / sample_size))

    # Configure AdamW + OneCycleLR over `total_steps` human-respondent samples
    trainable_params = list(kev_model.trainable_parameters())
    optimizer = torch.optim.AdamW(trainable_params, lr=lr, weight_decay=EPOCH_WEIGHT_DECAY, fused=True)
    if total_steps > 1:
        scheduler = torch.optim.lr_scheduler.OneCycleLR(
            optimizer,
            max_lr=lr,
            total_steps=total_steps,
            pct_start=0.1,
            div_factor=25.0,
            final_div_factor=1e4,
        )
    else:
        scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lambda _: 1.0)

    step_rows: list[dict] = []
    val_rows: list[dict] = [
        {"step": 0, "respondents_seen": 0, "train_elapsed_sec": 0.0, **step0_val_metrics}
    ]
    final_beta_df = step0_beta_df
    cum_train_sec = 0.0
    cum_eval_sec = 0.0
    respondents_seen = 0

    pbar = tqdm(range(1, total_steps + 1), total=total_steps, desc=f"S={sample_size} | lambda={beta_lambda:g}")
    for step_idx in pbar:
        start_idx = (step_idx - 1) * sample_size
        sample_items = epoch_items[start_idx : start_idx + sample_size]
        n_sample_resp = len(sample_items)
        respondents_seen += n_sample_resp

        torch.cuda.synchronize()
        t_step0 = time.perf_counter()
        optimizer.zero_grad(set_to_none=True)

        # 1. Single shared-prefix forward pass across all respondents in this sample (batched up to FWD_CHUNK_RESPONDENTS)
        sample_logit_tensors: list[torch.Tensor] = []
        with torch.autocast("cuda", dtype=torch.bfloat16):
            for c_start in range(0, n_sample_resp, FWD_CHUNK_RESPONDENTS):
                chunk_encs = [it["enc"] for it in sample_items[c_start : c_start + FWD_CHUNK_RESPONDENTS]]
                chunk_logits_lists = kev_model.forward(chunk_encs, shared_prefix=True)
                for resp_logits_list in chunk_logits_lists:
                    sample_logit_tensors.extend(z.to(dtype=torch.float64) for z in resp_logits_list)
        task_logits = torch.stack(sample_logit_tensors, dim=0)
        torch.cuda.synchronize()
        t_after_fwd = time.perf_counter()

        # 2. Stack `X_sample` and `y_sample` over all respondents in this step and fit sample-level OLS if rank == 9/9
        X_sample = torch.cat([it["X_resp"] for it in sample_items], dim=0)
        y_sample = torch.cat([it["y_resp"] for it in sample_items], dim=0)
        if X_sample.shape[0] >= X_sample.shape[1]:
            svals = torch.linalg.svdvals(X_sample)
            has_ols = float(svals[-1]) > OLS_SVD_RCOND * float(svals[0])
        else:
            has_ols = False

        if has_ols:
            A_sample = torch.linalg.pinv(X_sample)[1:].to(device=device)
            total_loss, kl_loss, beta_loss = compute_wu2025_loss(
                task_logits,
                y_sample,
                A_sample,
                beta_lambda=beta_lambda,
            )
            raw_beta_val = float(beta_loss.detach().cpu())
        else:
            A_sample = None
            beta_loss = None
            log_probs = torch.log_softmax(task_logits, dim=-1)
            kl_loss = torch.nn.functional.kl_div(log_probs, y_sample.view(-1, 2), reduction="batchmean")
            total_loss = kl_loss
            raw_beta_val = float("nan")

        kl_val = float(kl_loss.detach().cpu())
        total_val = float(total_loss.detach().cpu())
        torch.cuda.synchronize()
        t_after_loss = time.perf_counter()

        # 3. Single backward pass across the respondent sample, gradient clipping, AdamW update, and LR schedule step
        total_loss.backward()
        grad_norm = torch.nn.utils.clip_grad_norm_(trainable_params, 1.0)
        if not bool(torch.isfinite(grad_norm)):
            raise FloatingPointError(f"Non-finite gradient norm {float(grad_norm)} at step {step_idx}")
        grad_norm_val = float(grad_norm.detach().cpu())
        optimizer.step()
        if step_idx < total_steps:
            scheduler.step()
        optimizer.zero_grad(set_to_none=True)

        # Release per-step autograd references immediately so no computation graph lingers across steps or evaluations
        del sample_logit_tensors, task_logits, X_sample, y_sample, A_sample, total_loss, kl_loss, beta_loss
        torch.cuda.synchronize()
        t_after_bwd = time.perf_counter()

        fwd_ms = (t_after_fwd - t_step0) * 1000.0
        loss_ms = (t_after_loss - t_after_fwd) * 1000.0
        bwd_opt_ms = (t_after_bwd - t_after_loss) * 1000.0
        step_ms = (t_after_bwd - t_step0) * 1000.0
        resp_ms = step_ms / n_sample_resp
        cum_train_sec += t_after_bwd - t_step0

        step_rows.append(
            {
                "step": step_idx,
                "num_respondents": n_sample_resp,
                "respondents_seen": respondents_seen,
                "has_ols": has_ols,
                "kl_loss": kl_val,
                "raw_beta_loss": raw_beta_val,
                "weighted_beta_loss": beta_lambda * raw_beta_val if has_ols else 0.0,
                "total_loss": total_val,
                "grad_norm": grad_norm_val,
                "fwd_ms": fwd_ms,
                "loss_ms": loss_ms,
                "bwd_opt_ms": bwd_opt_ms,
                "step_ms": step_ms,
                "ms_per_respondent": resp_ms,
            }
        )

        # Update live progress bar with rolling (~300 respondents) statistical losses and step / per-respondent timings
        if step_idx % postfix_every == 0 or step_idx == total_steps:
            recent = step_rows[-rolling_steps:]
            recent_betas = [r["raw_beta_loss"] for r in recent if r["has_ols"]]
            pbar.set_postfix(
                kl=f"{float(np.mean([r['kl_loss'] for r in recent])):.3f}",
                beta=f"{float(np.mean(recent_betas)):.4f}" if recent_betas else "n/a",
                step_ms=f"{float(np.mean([r['step_ms'] for r in recent])):.1f}",
                resp_ms=f"{float(np.mean([r['ms_per_respondent'] for r in recent])):.1f}",
            )

        # 4. Evaluate population OLS approximation at quarter-epoch checkpoints and epoch end
        if step_idx % val_every == 0 or step_idx == total_steps:
            v_metrics, final_beta_df = validate_wu2025_population_ols(
                kev_model,
                wu2025_pop_bundle,
                A_pop,
                y_pop,
                human_pop_betas,
            )
            cum_eval_sec += v_metrics["eval_wall_sec"]
            val_rows.append(
                {
                    "step": step_idx,
                    "respondents_seen": respondents_seen,
                    "train_elapsed_sec": cum_train_sec,
                    **v_metrics,
                }
            )
            tqdm.write(
                f"[S={sample_size} | lambda={beta_lambda:g} | step {step_idx:4d}/{total_steps} ({respondents_seen}/{total_respondents} resp)] "
                f"pop_kl={v_metrics['pop_kl_loss']:.4f} | "
                f"pop_beta_mse={v_metrics['pop_beta_mse']:.5f} | "
                f"MAPE={v_metrics['pop_beta_mape_pct']:.2f}% | "
                f"rho={v_metrics['spearman_rho']:.3f} | "
                f"train={cum_train_sec:.1f}s | "
                f"eval_gpu={v_metrics['eval_gpu_sec']:.2f}s | "
                f"eval_wall={v_metrics['eval_wall_sec']:.2f}s"
            )

    total_wall_sec = time.perf_counter() - t_wall0
    step_df = pd.DataFrame(step_rows)
    val_history_df = pd.DataFrame(val_rows).set_index("step")
    ols_steps = step_df[step_df["has_ols"]]
    edge_window = max(1, round(0.15 * len(step_df)))          # first/last ~15% of epoch respondents regardless of sample_size
    ols_edge_window = max(1, round(0.15 * len(ols_steps)))

    # Summarize both statistical loss metrics and per-step / per-respondent / eval / wall-clock timing metrics
    summary = pd.Series(
        {
            "sample_size": sample_size,
            "beta_lambda": beta_lambda,
            "lr": lr,
            "steps_per_epoch": total_steps,
            "rank_deficient_steps": int((~step_df["has_ols"]).sum()),
            "train_kl_first15pct": float(step_df["kl_loss"].iloc[:edge_window].mean()),
            "train_kl_last15pct": float(step_df["kl_loss"].iloc[-edge_window:].mean()),
            "train_raw_beta_first15pct": float(ols_steps["raw_beta_loss"].iloc[:ols_edge_window].mean()),
            "train_raw_beta_last15pct": float(ols_steps["raw_beta_loss"].iloc[-ols_edge_window:].mean()),
            "train_weighted_beta_last15pct": float(ols_steps["weighted_beta_loss"].iloc[-ols_edge_window:].mean()),
            "train_total_first15pct": float(step_df["total_loss"].iloc[:edge_window].mean()),
            "train_total_last15pct": float(step_df["total_loss"].iloc[-edge_window:].mean()),
            "mean_grad_norm": float(step_df["grad_norm"].mean()),
            "mean_fwd_ms": float(step_df["fwd_ms"].mean()),
            "mean_loss_ms": float(step_df["loss_ms"].mean()),
            "mean_bwd_opt_ms": float(step_df["bwd_opt_ms"].mean()),
            "mean_step_ms": float(step_df["step_ms"].mean()),
            "mean_ms_per_respondent": (cum_train_sec * 1000.0) / total_respondents,
            "mean_eval_wall_sec": float(val_history_df["eval_wall_sec"].iloc[1:].mean()),
            "mean_eval_gpu_sec": float(val_history_df["eval_gpu_sec"].iloc[1:].mean()),
            "train_only_sec": cum_train_sec,
            "eval_only_sec": cum_eval_sec,
            "total_wall_sec": total_wall_sec,
        },
        name=f"S_{sample_size}_lambda_{beta_lambda:g}_summary",
    )
    display(summary.round(5).to_frame())
    display(val_history_df.round(5))

    return {
        "sample_size": sample_size,
        "beta_lambda": beta_lambda,
        "lr": lr,
        "summary": summary,
        "step_df": step_df,
        "val_history_df": val_history_df,
        "final_beta_df": final_beta_df,
    }


print("Defined scale_lr_for_sample_size(sample_size) and run_wu2025_epoch(sample_size, beta_lambda, lr)")

Defined scale_lr_for_sample_size(sample_size) and run_wu2025_epoch(sample_size, beta_lambda, lr)


In [12]:
# Number of human respondents sampled per optimization step (edit `SAMPLE_SIZE` here to run 8, 16, 32, etc.)
SAMPLE_SIZE = 16
SAMPLE_LR = scale_lr_for_sample_size(SAMPLE_SIZE)
sample_noise_info = analyze_sample_size_noise(sample_size=SAMPLE_SIZE, seed=EPOCH_SEED)

display(
    pd.Series(
        {
            "sample_size": SAMPLE_SIZE,
            "steps_per_epoch": sample_noise_info["steps_per_epoch"],
            "mean_option_rows_per_step": round(float(sample_noise_info["mean_option_rows"]), 2),
            "rank_deficient_steps": sample_noise_info["rank_deficient_steps"],
            "scaled_max_lr": SAMPLE_LR,
            "expected_target_noise_mse": round(float(sample_noise_info["target_noise_mse"]), 5),
            "expected_target_snr": round(float(sample_noise_info["snr_ratio"]), 2),
            "gradient_checkpointing": SAMPLE_SIZE > MAX_UNCHECKPOINTED_SAMPLE_SIZE,
        },
        name="selected_sample_size_config",
    ).to_frame()
)

,selected_sample_size_config
sample_size,16
steps_per_epoch,117
mean_option_rows_per_step,317.79
rank_deficient_steps,0
scaled_max_lr,0.00008
expected_target_noise_mse,0.00504
expected_target_snr,9.42
gradient_checkpointing,True


In [13]:
# Run 1 full epoch at `SAMPLE_SIZE` respondents/step with `beta_lambda = 0.0` (pure per-response KL/CE baseline)
experiment_runs[(SAMPLE_SIZE, 0.0)] = run_wu2025_epoch(
    sample_size=SAMPLE_SIZE,
    beta_lambda=0.0,
    lr=SAMPLE_LR,
)

S=16 | lambda=0:   0%|          | 0/117 [00:00<?, ?it/s]

[S=16 | lambda=0 | step   29/117 (464/1861 resp)] pop_kl=0.5711 | pop_beta_mse=0.00686 | MAPE=36.87% | rho=0.976 | train=64.0s | eval_gpu=32.30s | eval_wall=33.52s
[S=16 | lambda=0 | step   58/117 (928/1861 resp)] pop_kl=0.5398 | pop_beta_mse=0.00037 | MAPE=9.39% | rho=1.000 | train=129.0s | eval_gpu=33.80s | eval_wall=35.02s
[S=16 | lambda=0 | step   87/117 (1392/1861 resp)] pop_kl=0.5343 | pop_beta_mse=0.00020 | MAPE=7.69% | rho=1.000 | train=190.7s | eval_gpu=32.57s | eval_wall=33.75s
[S=16 | lambda=0 | step  116/117 (1856/1861 resp)] pop_kl=0.5327 | pop_beta_mse=0.00023 | MAPE=7.46% | rho=1.000 | train=251.3s | eval_gpu=31.84s | eval_wall=33.04s
[S=16 | lambda=0 | step  117/117 (1861/1861 resp)] pop_kl=0.5327 | pop_beta_mse=0.00023 | MAPE=7.46% | rho=1.000 | train=252.0s | eval_gpu=33.31s | eval_wall=34.50s


,S_16_lambda_0_summary
sample_size,16.00000
beta_lambda,0.00000
lr,0.00008
steps_per_epoch,117.00000
rank_deficient_steps,0.00000
train_kl_first15pct,0.64236
train_kl_last15pct,0.53854
train_raw_beta_first15pct,0.01800
train_raw_beta_last15pct,0.01048
train_weighted_beta_last15pct,0.00000


,respondents_seen,train_elapsed_sec,pop_kl_loss,pop_beta_mse,pop_beta_mae,pop_beta_mape_pct,spearman_rho,eval_prep_sec,eval_gpu_sec,eval_ols_ms,eval_wall_sec,eval_ms_per_task
step,,,,,,,,,,,,
0,0,0.00000,1.06993,0.05118,0.20925,114.13144,-0.21429,0.20566,33.38287,5.35743,35.49985,1.90952
29,464,64.01733,0.57112,0.00686,0.07259,36.86888,0.97619,0.01177,32.29680,6.22249,33.51761,1.80289
58,928,129.02449,0.53978,0.00037,0.01681,9.39142,1.00000,0.01177,33.79877,6.40258,35.01521,1.88345
87,1392,190.65247,0.53431,0.00020,0.01284,7.69400,1.00000,0.01100,32.56571,6.19193,33.75032,1.81541
116,1856,251.28247,0.53272,0.00023,0.01258,7.45785,1.00000,0.01089,31.84356,6.34223,33.03543,1.77696
117,1861,251.95613,0.53275,0.00023,0.01258,7.45788,1.00000,0.01124,33.31077,6.09072,34.50436,1.85597


In [14]:
# Run 1 full epoch at `SAMPLE_SIZE` respondents/step with `beta_lambda = 1.0` (mild sample OLS beta weight)
experiment_runs[(SAMPLE_SIZE, 1.0)] = run_wu2025_epoch(
    sample_size=SAMPLE_SIZE,
    beta_lambda=1.0,
    lr=SAMPLE_LR,
)

S=16 | lambda=1:   0%|          | 0/117 [00:00<?, ?it/s]

[S=16 | lambda=1 | step   29/117 (464/1861 resp)] pop_kl=0.5763 | pop_beta_mse=0.00870 | MAPE=41.17% | rho=0.976 | train=65.8s | eval_gpu=33.52s | eval_wall=34.71s
[S=16 | lambda=1 | step   58/117 (928/1861 resp)] pop_kl=0.5409 | pop_beta_mse=0.00012 | MAPE=5.95% | rho=0.976 | train=131.0s | eval_gpu=32.69s | eval_wall=33.84s
[S=16 | lambda=1 | step   87/117 (1392/1861 resp)] pop_kl=0.5365 | pop_beta_mse=0.00022 | MAPE=8.18% | rho=1.000 | train=195.3s | eval_gpu=33.42s | eval_wall=34.68s
[S=16 | lambda=1 | step  116/117 (1856/1861 resp)] pop_kl=0.5348 | pop_beta_mse=0.00009 | MAPE=3.86% | rho=1.000 | train=260.3s | eval_gpu=33.45s | eval_wall=34.64s
[S=16 | lambda=1 | step  117/117 (1861/1861 resp)] pop_kl=0.5348 | pop_beta_mse=0.00009 | MAPE=3.89% | rho=1.000 | train=261.0s | eval_gpu=33.68s | eval_wall=34.85s


,S_16_lambda_1_summary
sample_size,16.00000
beta_lambda,1.00000
lr,0.00008
steps_per_epoch,117.00000
rank_deficient_steps,0.00000
train_kl_first15pct,0.64005
train_kl_last15pct,0.53929
train_raw_beta_first15pct,0.01662
train_raw_beta_last15pct,0.01022
train_weighted_beta_last15pct,0.01022


,respondents_seen,train_elapsed_sec,pop_kl_loss,pop_beta_mse,pop_beta_mae,pop_beta_mape_pct,spearman_rho,eval_prep_sec,eval_gpu_sec,eval_ols_ms,eval_wall_sec,eval_ms_per_task
step,,,,,,,,,,,,
0,0,0.00000,1.06993,0.05118,0.20925,114.13144,-0.21429,0.20566,33.38287,5.35743,35.49985,1.90952
29,464,65.81507,0.57630,0.00870,0.08174,41.16556,0.97619,0.01126,33.51596,6.23066,34.71172,1.86712
58,928,131.01653,0.54086,0.00012,0.00890,5.94953,0.97619,0.01093,32.68886,6.20107,33.84133,1.82031
87,1392,195.29432,0.53651,0.00022,0.01265,8.18200,1.00000,0.01147,33.42246,6.27835,34.68277,1.86557
116,1856,260.32068,0.53480,0.00009,0.00738,3.86392,1.00000,0.01131,33.45310,6.12279,34.63579,1.86304
117,1861,260.98318,0.53481,0.00009,0.00742,3.89011,1.00000,0.01121,33.67675,6.03150,34.84701,1.87440


In [15]:
# Run 1 full epoch at `SAMPLE_SIZE` respondents/step with `beta_lambda = 5.0` (moderate sample OLS beta weight)
experiment_runs[(SAMPLE_SIZE, 5.0)] = run_wu2025_epoch(
    sample_size=SAMPLE_SIZE,
    beta_lambda=5.0,
    lr=SAMPLE_LR,
)

S=16 | lambda=5:   0%|          | 0/117 [00:00<?, ?it/s]

[S=16 | lambda=5 | step   29/117 (464/1861 resp)] pop_kl=0.5828 | pop_beta_mse=0.00820 | MAPE=33.17% | rho=1.000 | train=65.8s | eval_gpu=32.67s | eval_wall=33.82s
[S=16 | lambda=5 | step   58/117 (928/1861 resp)] pop_kl=0.5423 | pop_beta_mse=0.00023 | MAPE=8.56% | rho=0.976 | train=127.5s | eval_gpu=31.55s | eval_wall=32.70s
[S=16 | lambda=5 | step   87/117 (1392/1861 resp)] pop_kl=0.5390 | pop_beta_mse=0.00063 | MAPE=12.08% | rho=1.000 | train=188.4s | eval_gpu=32.50s | eval_wall=33.64s
[S=16 | lambda=5 | step  116/117 (1856/1861 resp)] pop_kl=0.5375 | pop_beta_mse=0.00017 | MAPE=4.73% | rho=1.000 | train=250.4s | eval_gpu=31.41s | eval_wall=32.48s
[S=16 | lambda=5 | step  117/117 (1861/1861 resp)] pop_kl=0.5375 | pop_beta_mse=0.00017 | MAPE=4.72% | rho=1.000 | train=251.0s | eval_gpu=31.79s | eval_wall=32.89s


,S_16_lambda_5_summary
sample_size,16.00000
beta_lambda,5.00000
lr,0.00008
steps_per_epoch,117.00000
rank_deficient_steps,0.00000
train_kl_first15pct,0.64498
train_kl_last15pct,0.54144
train_raw_beta_first15pct,0.01620
train_raw_beta_last15pct,0.01026
train_weighted_beta_last15pct,0.05129


,respondents_seen,train_elapsed_sec,pop_kl_loss,pop_beta_mse,pop_beta_mae,pop_beta_mape_pct,spearman_rho,eval_prep_sec,eval_gpu_sec,eval_ols_ms,eval_wall_sec,eval_ms_per_task
step,,,,,,,,,,,,
0,0,0.00000,1.06993,0.05118,0.20925,114.13144,-0.21429,0.20566,33.38287,5.35743,35.49985,1.90952
29,464,65.83237,0.58280,0.00820,0.07108,33.16611,1.00000,0.01093,32.67417,6.20260,33.81654,1.81897
58,928,127.54836,0.54228,0.00023,0.01367,8.56268,0.97619,0.01134,31.55364,6.73528,32.69595,1.75870
87,1392,188.38175,0.53905,0.00063,0.02207,12.08249,1.00000,0.01144,32.49578,6.98414,33.63825,1.80938
116,1856,250.43394,0.53745,0.00017,0.00983,4.73185,1.00000,0.01091,31.41230,5.95631,32.47848,1.74700
117,1861,251.04617,0.53746,0.00017,0.00982,4.72389,1.00000,0.01090,31.78843,5.93605,32.89050,1.76916


In [16]:
# Run 1 full epoch at `SAMPLE_SIZE` respondents/step with `beta_lambda = 20.0` (strong sample OLS beta weight)
experiment_runs[(SAMPLE_SIZE, 20.0)] = run_wu2025_epoch(
    sample_size=SAMPLE_SIZE,
    beta_lambda=20.0,
    lr=SAMPLE_LR,
)

S=16 | lambda=20:   0%|          | 0/117 [00:00<?, ?it/s]

[S=16 | lambda=20 | step   29/117 (464/1861 resp)] pop_kl=0.5906 | pop_beta_mse=0.00206 | MAPE=24.98% | rho=0.976 | train=66.7s | eval_gpu=34.48s | eval_wall=35.58s
[S=16 | lambda=20 | step   58/117 (928/1861 resp)] pop_kl=0.5524 | pop_beta_mse=0.00119 | MAPE=10.22% | rho=0.976 | train=133.5s | eval_gpu=33.80s | eval_wall=34.93s
[S=16 | lambda=20 | step   87/117 (1392/1861 resp)] pop_kl=0.5488 | pop_beta_mse=0.00073 | MAPE=11.38% | rho=1.000 | train=199.0s | eval_gpu=33.59s | eval_wall=34.70s
[S=16 | lambda=20 | step  116/117 (1856/1861 resp)] pop_kl=0.5428 | pop_beta_mse=0.00075 | MAPE=11.30% | rho=0.976 | train=263.9s | eval_gpu=33.31s | eval_wall=34.43s
[S=16 | lambda=20 | step  117/117 (1861/1861 resp)] pop_kl=0.5427 | pop_beta_mse=0.00075 | MAPE=11.30% | rho=0.976 | train=264.5s | eval_gpu=33.57s | eval_wall=34.69s


,S_16_lambda_20_summary
sample_size,16.00000
beta_lambda,20.00000
lr,0.00008
steps_per_epoch,117.00000
rank_deficient_steps,0.00000
train_kl_first15pct,0.64586
train_kl_last15pct,0.54667
train_raw_beta_first15pct,0.01466
train_raw_beta_last15pct,0.01101
train_weighted_beta_last15pct,0.22027


,respondents_seen,train_elapsed_sec,pop_kl_loss,pop_beta_mse,pop_beta_mae,pop_beta_mape_pct,spearman_rho,eval_prep_sec,eval_gpu_sec,eval_ols_ms,eval_wall_sec,eval_ms_per_task
step,,,,,,,,,,,,
0,0,0.00000,1.06993,0.05118,0.20925,114.13144,-0.21429,0.20566,33.38287,5.35743,35.49985,1.90952
29,464,66.71099,0.59057,0.00206,0.04248,24.98489,0.97619,0.01078,34.48285,6.19053,35.58084,1.91387
58,928,133.48610,0.55236,0.00119,0.02392,10.22324,0.97619,0.01195,33.80204,6.06149,34.92690,1.87870
87,1392,198.95266,0.54878,0.00073,0.02221,11.37943,1.00000,0.01076,33.59184,6.03874,34.69588,1.86627
116,1856,263.87601,0.54276,0.00075,0.02043,11.30039,0.97619,0.01053,33.30849,6.00792,34.43418,1.85220
117,1861,264.54640,0.54274,0.00075,0.02044,11.30017,0.97619,0.01066,33.56657,5.90774,34.68537,1.86571


In [17]:
# Compare statistical metrics and step / respondent / eval / epoch timing metrics across all completed runs
if not experiment_runs:
    raise ValueError("Run at least one `run_wu2025_epoch` cell first so `experiment_runs` is populated")
ran_keys = sorted(experiment_runs.keys())

compare_rows = [
    {
        "run": "step_0 (untrained)",
        "sample_size": 0,
        "beta_lambda": float("nan"),
        "steps_per_epoch": 0,
        "train_kl_last15pct": float("nan"),
        "train_raw_beta_last15pct": float("nan"),
        "pop_kl_loss": step0_val_metrics["pop_kl_loss"],
        "pop_beta_mse": step0_val_metrics["pop_beta_mse"],
        "pop_beta_mae": step0_val_metrics["pop_beta_mae"],
        "pop_beta_mape_pct": step0_val_metrics["pop_beta_mape_pct"],
        "spearman_rho": step0_val_metrics["spearman_rho"],
        "mean_step_ms": float("nan"),
        "mean_ms_per_respondent": float("nan"),
        "mean_eval_wall_sec": step0_val_metrics["eval_wall_sec"],
        "train_only_sec": 0.0,
        "eval_only_sec": step0_val_metrics["eval_wall_sec"],
        "total_wall_sec": step0_val_metrics["eval_wall_sec"],
    }
]
for s_size, lam in ran_keys:
    run = experiment_runs[(s_size, lam)]
    s = run["summary"]
    final_val = run["val_history_df"].iloc[-1]
    compare_rows.append(
        {
            "run": f"S={s_size}, lambda={lam:g}",
            "sample_size": s_size,
            "beta_lambda": lam,
            "steps_per_epoch": int(s["steps_per_epoch"]),
            "train_kl_last15pct": float(s["train_kl_last15pct"]),
            "train_raw_beta_last15pct": float(s["train_raw_beta_last15pct"]),
            "pop_kl_loss": float(final_val["pop_kl_loss"]),
            "pop_beta_mse": float(final_val["pop_beta_mse"]),
            "pop_beta_mae": float(final_val["pop_beta_mae"]),
            "pop_beta_mape_pct": float(final_val["pop_beta_mape_pct"]),
            "spearman_rho": float(final_val["spearman_rho"]),
            "mean_step_ms": float(s["mean_step_ms"]),
            "mean_ms_per_respondent": float(s["mean_ms_per_respondent"]),
            "mean_eval_wall_sec": float(s["mean_eval_wall_sec"]),
            "train_only_sec": float(s["train_only_sec"]),
            "eval_only_sec": float(s["eval_only_sec"]),
            "total_wall_sec": float(s["total_wall_sec"]),
        }
    )

comparison_df = pd.DataFrame(compare_rows).set_index("run")
display(comparison_df.round(5))

# Compare the 8 learned population OLS betas against the ground-truth human population OLS betas
pop_betas_compare_df = pd.DataFrame(
    {
        "human_pop_beta": step0_beta_df["human_pop_beta"],
        "step_0": step0_beta_df["model_pop_beta"],
        **{
            f"S={s_size}, lambda={lam:g}": experiment_runs[(s_size, lam)]["final_beta_df"]["model_pop_beta"]
            for s_size, lam in ran_keys
        },
    }
)
display(pop_betas_compare_df.round(4))

,sample_size,beta_lambda,steps_per_epoch,train_kl_last15pct,train_raw_beta_last15pct,pop_kl_loss,pop_beta_mse,pop_beta_mae,pop_beta_mape_pct,spearman_rho,mean_step_ms,mean_ms_per_respondent,mean_eval_wall_sec,train_only_sec,eval_only_sec,total_wall_sec
run,,,,,,,,,,,,,,,,
step_0 (untrained),0,NaN,0,NaN,NaN,1.06993,0.05118,0.20925,114.13144,-0.21429,NaN,NaN,35.49985,0.00000,35.49985,35.49985
"S=16, lambda=0",16,0.0,117,0.53854,0.01048,0.53275,0.00023,0.01258,7.45788,1.00000,2153.47120,135.38750,33.96459,251.95613,169.82293,422.27105
"S=16, lambda=1",16,1.0,117,0.53929,0.01022,0.53481,0.00009,0.00742,3.89011,1.00000,2230.62548,140.23814,34.54372,260.98318,172.71862,434.16813
"S=16, lambda=5",16,5.0,117,0.54144,0.01026,0.53746,0.00017,0.00982,4.72389,1.00000,2145.69373,134.89853,33.10395,251.04617,165.51973,417.01760
"S=16, lambda=20",16,20.0,117,0.54667,0.01101,0.54274,0.00075,0.02044,11.30017,0.97619,2261.08038,142.15282,34.86463,264.54640,174.32317,439.29549


,human_pop_beta,step_0,"S=16, lambda=0","S=16, lambda=1","S=16, lambda=5","S=16, lambda=20"
term,,,,,,
Monthly bill: $110 vs $105,-0.0969,-0.0068,-0.0822,-0.0960,-0.0935,-0.0937
Monthly bill: $115 vs $105,-0.1779,-0.0122,-0.1612,-0.1778,-0.1754,-0.1773
Monthly bill: $120 vs $105,-0.2894,-0.0171,-0.2733,-0.2783,-0.2768,-0.2734
Monthly bill: $125 vs $105,-0.3832,-0.0248,-0.3548,-0.3640,-0.3685,-0.3661
Percentage of renewables: 30% vs 10%,0.0973,-0.0075,0.0898,0.0853,0.0943,0.1177
Percentage of renewables: 50% vs 10%,0.1706,-0.0149,0.1704,0.1616,0.1779,0.2057
Percentage of renewables: 100% vs 10%,0.2421,-0.0398,0.2385,0.2364,0.2718,0.3025
Locally produced renewables: Yes vs No,0.1042,-0.1109,0.0909,0.1028,0.1095,0.0936
